# Tutorial 02 - Images
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-Media/vco/blob/main/02_Images.ipynb)

## Dr. David C. Schedl

Note: this tutorial is geared towards students **experienced in general programming** and aims to introduce you to image representation and color spaces with OpenCV.

Useful links:
* OpenCV Tutorials: https://docs.opencv.org/master/d9/df8/tutorial_root.html
* Image Processing in Python: https://github.com/xn2333/OpenCV/blob/master/Seminar_Image_Processing_in_Python.ipynb

# Contents

- [Initialization](#Initialization)
- [OpenCV Basics](#OpenCV-Basics)
    - Reading images
    - Channels and Image Formats
    - Displaying with Matplotlib
    - Color Channels
- [Examples from Slides](#Examples-from-Slides)
    - Bayer Filter
    - Naive Image Resizing
- [Color Spaces](#Color-Spaces)
    - HSV
    - Lab*
    - Comparison
- [Exercises](#Exercises)

# Initialization <a id="Initialization"></a>

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

We work with images today. Let's download some sample images.

Image sources:
* A picture of a cat from the [cv_data](https://github.com/Digital-Media/cv_data) repository.
* A picture of Van Gogh from the same repository.

In [ ]:
!curl -o "cat.jpg" "https://raw.githubusercontent.com/Digital-Media/cv_data/main/example_images/cat.jpg" --silent
!curl -o "gogh.jpg" "https://raw.githubusercontent.com/Digital-Media/cv_data/main/example_images/gogh.jpg" --silent

# OpenCV Basics <a id="OpenCV-Basics"></a>

OpenCV is an extremely popular computer vision library built in C++. The Python API exposes its functionality using NumPy arrays.

## Reading Images

Use `cv2.imread()` to read an image from a filepath. The result is a NumPy array.

In [ ]:
image = cv2.imread("gogh.jpg")
print(f"Type: {type(image)}, Shape: {image.shape}, Dtype: {image.dtype}")

## Channels and Image Formats

The shape of a color image is `(height, width, channels)`. Height comes first because OpenCV treats images as rows and columns.

Each pixel is represented by 3 `uint8` values (0–255). **Important:** OpenCV stores channels in **BGR** order (blue, green, red), not RGB!

In [ ]:
print(f"Image shape: {image.shape}")
print(f"Pixel at (0,0) [BGR]: {image[0, 0]}")

## Displaying with Matplotlib

Since images are NumPy arrays, we can plot them directly with Matplotlib. Let's try:

In [ ]:
plt.imshow(image)
plt.axis("off")
plt.title("Displayed directly \u2014 colors are wrong!")
plt.show()

**The colors are wrong!** Matplotlib expects **RGB**, but OpenCV stores images as **BGR**.

We use `cv2.cvtColor` to convert:

In [ ]:
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.imshow(image_rgb)
plt.axis("off")
plt.title("After BGR \u2192 RGB conversion")
plt.show()

To avoid repeating this conversion, here is a helper function:

In [ ]:
def imshow(image, *args, **kwargs):
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    plt.imshow(image, *args, **kwargs)
    plt.axis("off")
    plt.show()

## Color Channels

Let's separate and visualize the individual BGR channels. Each channel is displayed tinted in its actual color.

In [ ]:
b, g, r = cv2.split(image)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (RGB)")

channels = [(r, 0, "Red"), (g, 1, "Green"), (b, 2, "Blue")]
for ax, (ch, idx, name) in zip(axes[1:], channels):
    tinted = np.zeros((*ch.shape, 3), dtype=np.uint8)
    tinted[:, :, idx] = ch
    ax.imshow(tinted)
    ax.set_title(name)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Examples from Slides

## Bayer Filter

Image sensors have a Bayer filter on the sensor elements (pixels). So every 2nd pixel in even rows is red, every 2nd pixel in odd rows is blue, and every 2nd pixel in all rows contains green:

|     |  0  |  1  |
|---  | --- | --- |
|**0**| R   | G   |
|**1**| G   | B   |

Let's create a `raw` image with such a Bayer pattern from our loaded images. We can use masking and slicing.

In [ ]:
red_mask = np.zeros(shape=image.shape[0:2], dtype=bool)
green_mask, blue_mask = red_mask.copy(), red_mask.copy()
red_mask[0::2, 0::2] = True
blue_mask[1::2, 1::2] = True
green_mask[0::2, 1::2] = True
green_mask[1::2, 0::2] = True

rsize = np.ceil(np.asarray(red_mask.shape) / 2.0).astype(int)
reds = image[:, :, 2][red_mask].reshape(rsize)

bsize = np.floor(np.asarray(blue_mask.shape) / 2.0).astype(int)
blues = image[:, :, 0][blue_mask].reshape(bsize)

bayer = np.zeros(shape=(*image.shape[0:2], 3), dtype=image.dtype)
bayer[:, :, 0][blue_mask] = blues.flatten()
bayer[:, :, 2][red_mask] = reds.flatten()
bayer[:, :, 1][green_mask] = image[:, :, 1][green_mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(bayer[:16, :16], cv2.COLOR_BGR2RGB), interpolation="nearest")
axes[0].set_title("Bayer Pattern (zoomed 16×16)")
axes[1].imshow(cv2.cvtColor(bayer, cv2.COLOR_BGR2RGB))
axes[1].set_title("Full Bayer Image")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

The pattern above shows what most color image sensors record. The process of converting such a Bayer pattern into a color image is called **demosaicing**.

## Naive Image Resizing

… by simply throwing away rows and columns (see lecture slides).
We can use Python's slicing notation to do so.

In [ ]:
def downsample(img):
    return img[0::2, 0::2]

half = downsample(image)
quad = downsample(half)
eighth = downsample(quad)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original ({image.shape[1]}×{image.shape[0]})")
axes[1].imshow(cv2.cvtColor(eighth, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Naive 1/8 ({eighth.shape[1]}×{eighth.shape[0]})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Dropping pixels is not a good way to downscale images. It leads to severe **aliasing** problems (see lecture slides).

Let's compare to OpenCV's built-in `resize` function:

In [ ]:
smooth = cv2.resize(image, eighth.shape[1::-1], interpolation=cv2.INTER_AREA)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(eighth, cv2.COLOR_BGR2RGB))
axes[0].set_title("Naive downsample (aliasing)")
axes[1].imshow(cv2.cvtColor(smooth, cv2.COLOR_BGR2RGB))
axes[1].set_title("OpenCV resize (no aliasing)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Color Spaces <a id="Color-Spaces"></a>

Different color spaces encode color information in different ways — each useful for different tasks.
OpenCV provides `cv2.cvtColor()` to convert between them.

## HSV — Hue, Saturation, Value

- **Hue** (0–179 in OpenCV): the color type
- **Saturation** (0–255): color purity
- **Value** (0–255): brightness

HSV separates *what color* from *how bright*, making it ideal for **color-based segmentation**.

In [ ]:
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(h, cmap="hsv", vmin=0, vmax=179)
axes[1].set_title("Hue")
axes[2].imshow(s, cmap="gray")
axes[2].set_title("Saturation")
axes[3].imshow(v, cmap="gray")
axes[3].set_title("Value")
for ax in axes:
    ax.axis("off")
plt.suptitle("HSV Color Space", fontsize=14)
plt.tight_layout()
plt.show()

## Lab* (CIELAB)

- **L***: Lightness
- **a***: Green ↔ Red axis
- **b***: Blue ↔ Yellow axis

Lab* is **perceptually uniform**: equal numerical differences ≈ equal perceived color differences.

In [ ]:
lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
l_ch, a_ch, b_ch = cv2.split(lab)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(l_ch, cmap="gray")
axes[1].set_title("L* (Lightness)")
axes[2].imshow(a_ch, cmap="RdYlGn_r")
axes[2].set_title("a* (Green \u2194 Red)")
axes[3].imshow(b_ch, cmap="PiYG_r")
axes[3].set_title("b* (Blue \u2194 Yellow)")
for ax in axes:
    ax.axis("off")
plt.suptitle("Lab* Color Space", fontsize=14)
plt.tight_layout()
plt.show()

## Color Space Comparison

Let's compare all color spaces side by side — showing the individual channels of RGB, HSV, and Lab* for both sample images.

In [ ]:
cat = cv2.imread("cat.jpg")

def show_color_spaces(img, title):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    fig.suptitle(title, fontsize=16)

    # RGB
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_ylabel("RGB", fontsize=12)
    for i, name in enumerate(["Red", "Green", "Blue"]):
        axes[0, i + 1].imshow(rgb[:, :, i], cmap="gray")
        axes[0, i + 1].set_title(name)

    # HSV
    axes[1, 0].imshow(rgb)
    axes[1, 0].set_ylabel("HSV", fontsize=12)
    hsv_cfg = [("Hue", "hsv"), ("Saturation", "gray"), ("Value", "gray")]
    for i, (name, cmap) in enumerate(hsv_cfg):
        axes[1, i + 1].imshow(hsv[:, :, i], cmap=cmap)
        axes[1, i + 1].set_title(name)

    # Lab
    axes[2, 0].imshow(rgb)
    axes[2, 0].set_ylabel("Lab*", fontsize=12)
    lab_cfg = [("L*", "gray"), ("a*", "RdYlGn_r"), ("b*", "PiYG_r")]
    for i, (name, cmap) in enumerate(lab_cfg):
        axes[2, i + 1].imshow(lab[:, :, i], cmap=cmap)
        axes[2, i + 1].set_title(name)

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_color_spaces(image, "Van Gogh \u2014 Color Spaces")
show_color_spaces(cat, "Cat \u2014 Color Spaces")

# Exercises <a id="Exercises"></a>

## Exercise 1

**Grayscale:** Color is nice, but monochrome images are also very appealing.
Displaying a single color channel does not really look nice. So we need a weighted sum of all channels.
Typical weights to convert from RGB to grayscale are:
> $0.2989 \cdot R + 0.5870 \cdot G + 0.1140 \cdot B$

**(a)** Load the image `gogh.jpg`. Convert it to grayscale and display it. Don't forget that channels are BGR.

In [ ]:
# Solution (a)


## Exercise 2

**Gamma Curve:** 8-bit images are stored non-linearly (like our perception).
A common function used for this non-linear mapping is the gamma curve: $y = x^\gamma$, where $x$ are the linear values in the range \[0, 1\].

**(a)** Linearize the image `gogh.jpg` (apply $\gamma = 2.2$) and display it.

**(b)** Apply different $\gamma$ values (e.g., 0.5, 1.0, 1.5, 2.5) and show the results side by side using `plt.subplots`.

In [ ]:
# Solution


## Exercise 3

**Color-Based Fruit Separation:** The [Fruits-360](https://github.com/fruits-360/fruits-360-100x100) dataset contains 100×100 images of various fruits on a white background.

**(a)** Run the cell below to download the dataset and load sample fruit images.

**(b)** Convert the fruit images to **HSV** and **Lab\*** color spaces. Display the individual channels for each fruit side by side (similar to the color space comparison above). Which channels best distinguish the different fruits?

**(c)** Try to **separate a fruit from its background** using simple color thresholding in HSV. Use `cv2.inRange()` to create a binary mask based on saturation (the white background has low saturation).

**Hint:**
```python
hsv_fruit = cv2.cvtColor(fruit_img, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv_fruit, np.array([0, 40, 40]), np.array([179, 255, 255]))
result = cv2.bitwise_and(fruit_img, fruit_img, mask=mask)
```

**(d)** Can you also distinguish a **red apple** from an **orange** by thresholding on the **Hue** channel? What about a **banana** vs. a **lemon**? Which color space works best for each case?

In [ ]:
from pathlib import Path

!git clone --depth 1 https://github.com/fruits-360/fruits-360-100x100.git 2>/dev/null || echo "Dataset already cloned."

TRAIN_DIR = Path("fruits-360-100x100") / "Training"

def load_fruit(class_name, index=0):
    """Load a single fruit image (BGR) by class name and index."""
    cls_dir = TRAIN_DIR / class_name
    return cv2.imread(str(sorted(cls_dir.glob("*.jpg"))[index]))

fruit_names = ["Apple Red 1", "Banana 1", "Kiwi 1", "Orange 1", "Blueberry 1"]
fruits = {name: load_fruit(name) for name in fruit_names}

fig, axes = plt.subplots(1, len(fruits), figsize=(15, 3))
for ax, (name, img) in zip(axes, fruits.items()):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.suptitle("Fruits-360 Samples", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Solution
